In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from numpy import log
from openpmd_viewer import OpenPMDTimeSeries
from scipy.constants import alpha, c, eV, m_e, pi

# useful constants
GeV = 1e9 * eV
gamma_em = 0.5772156649015  # Euler-Mascheroni

# electron energy
gamma0 = 125 * GeV / (m_e * c**2)

### Theory

In [ ]:
# fractional energy of the virtual photon wrt electron energy
ymin = 0.0001 / gamma0
print("ymin", ymin)
y = np.geomspace(ymin, 1, 1000)

# spectrum of virtual photons for one electron
A = log(4) - 2 * gamma_em - 1
dN_dy = alpha / pi / y * (-2 * log(y) + A)
print(A, np.exp(A / 2))


# number of virtual photons for one electron
N = alpha / pi * (-A * log(ymin) + log(ymin) ** 2)

### WarpX

In [ ]:
# set up figure
fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(5, 4), dpi=100)


ax.plot(y, dN_dy / N, color="black")

series = OpenPMDTimeSeries("./diags/diag1/")

uz, w, ids = series.get_particle(
    ["uz", "w", "id"], species="virtual_photons", iteration=1
)
print(w)
for uid in np.unique(ids):
    y_wx = uz[ids == uid] * c / (m_e * c**2 * gamma0)
    w_wx = w[ids == uid]
    H, b = np.histogram(y_wx, bins=10, weights=w_wx)
    b = 0.5 * (b[1:] + b[:-1])
    db = b[1] - b[0]

    ax.plot(b, H / db / np.sum(w_wx), label=f"uid {uid}")

ax.set_yscale("log")
plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=2, nrows=1, figsize=(9, 5), dpi=100)

GeV = 1e9 * eV
gamma0 = 125 * GeV / (m_e * c**2)

print("gamma0", gamma0)
gamma_em = 0.5772156649015

A = log(4) - 2 * gamma_em - 1

ymin = 0.0001 / gamma0
y = np.linspace(ymin, 1, 100)
dN_dy = alpha / pi / y * (-2 * log(y) + A)

print("ymin", ymin)
N = alpha / pi * (-A * log(ymin) + log(ymin) ** 2)
norm = np.trapezoid(dN_dy, y)

ax[0].semilogy(y, dN_dy, color="black", lw=4)

lns4 = log(gamma0**2)
N_xs_gp = alpha / pi * (-log(ymin) * (-log(ymin) + lns4))
print(N_xs_gp)

it = 1
series = OpenPMDTimeSeries("./diags/diag1/")

uz, w, ids = series.get_particle(
    ["uz", "w", "id"], species="virtual_photons", iteration=it
)
for uid in np.unique(ids):
    H, b = np.histogram(uz[ids == uid] * c / GeV, bins=10, weights=w[ids == uid])
    b = 0.5 * (b[1:] + b[:-1])
    db = b[1] - b[0]
    # print(np.min(uz[ids == uid]), np.max(uz[ids == uid]))
    ax[1].semilogy(b, H / db, label=f"uid {uid}")

    norm = np.trapezoid(H / db, b)
    # ax[0].semilogy(b, H/db/norm, label=f"uid {uid}")

plt.show()

In [ ]:
"""
fig, ax = plt.subplots(ncols=1, nrows=7, figsize=(5, 24), dpi=100)

it = 1
series = OpenPMDTimeSeries("./diags/diag1/")


x, y, z, ux, uy, uz, w, ids = series.get_particle(
    ["x", "y", "z", "ux", "uy", "uz", "w", "id"], species="beam", iteration=it
)
ax[0].plot(ids, x, ".", ms=20, color="black")
ax[1].plot(ids, y, ".", ms=20, color="black")
ax[2].plot(ids, z, ".", ms=20, color="black")
ax[3].plot(ids, ux * m_e * c, ".", ms=20, color="black")
ax[4].plot(ids, uy * m_e * c, ".", ms=20, color="black")
ax[5].plot(ids, uz * m_e * c, ".", ms=20, color="black")
ax[6].hist(ids, zorder=2, edgecolor="black", facecolor="none", linewidth=2)
print(len(w), len(w) * 21)

x, y, z, ux, uy, uz, w, ids = series.get_particle(
    ["x", "y", "z", "ux", "uy", "uz", "w", "id"],
    species="virtual_photons",
    iteration=it,
)
ax[0].plot(ids, x, ".", ms=10, color="dodgerblue")
ax[1].plot(ids, y, ".", ms=10, color="dodgerblue")
ax[2].plot(ids, z, ".", ms=10, color="dodgerblue")
ax[3].plot(ids, ux, ".", ms=10, color="dodgerblue")
ax[4].plot(ids, uy, ".", ms=10, color="dodgerblue")
ax[5].plot(ids, uz, ".", ms=10, color="dodgerblue")
ax[6].hist(ids, zorder=1, edgecolor="dodgerblue", facecolor="none", linewidth=2)
print(len(w))
plt.show()
"""